# 01. 행정안전부 공공서비스(혜택) 적재

> **목적**: 행정안전부 `대한민국 공공서비스(혜택)` API (`api.odcloud.kr/api/gov24/v3`)에서
> 서비스 목록 + supportConditions(자격조건)를 받아 정형화 후 Supabase에 적재합니다.

## 전제 조건
- `schema.sql`이 Supabase Dashboard → SQL Editor에서 실행되어 테이블 3개가 생성된 상태
- `.env`에 `SUPABASE_URL`, `SUPABASE_SERVICE_KEY`, `PUBLIC_SERVICE_API_KEY` 입력됨

## 적재 흐름

```
[GET] /gov24/v3/serviceList        ← 페이지네이션 ~1,100건
       │
       ▼
   normalize_public_service()      ← 정형화
       │
       ├──► welfare_services UPSERT
       ▼
[GET] /gov24/v3/supportConditions  ← 서비스ID 단건 조회
       │
       ▼
   normalize_support_conditions()  ← JA**** 코드 → boolean 컬럼
       │
       └──► welfare_support_conditions UPSERT
```

## 소요 시간
- 첫 실행 ~10분 (서비스 1,100건 × 0.15초 조건 조회)
- 증분 갱신 ~2분 (upsert로 변경 분만 반영)


## 1. 환경 설정

In [1]:
# !pip install supabase requests python-dotenv


In [1]:
from dotenv import load_dotenv
from pathlib import Path
import os, json, time
from datetime import datetime, timezone
from urllib.parse import unquote
import requests

load_dotenv(".env", override=True)

REQUIRED = ["PUBLIC_SERVICE_API_KEY", "SUPABASE_URL", "SUPABASE_SERVICE_KEY"]
missing = [k for k in REQUIRED if not os.getenv(k) or str(os.getenv(k,"")).startswith(("발급","https://your"))]
if missing:
    print(f"⚠️ 필수 키 누락: {missing}")
else:
    print("✅ 필수 키 OK")

from supabase import create_client, Client
SB: Client = create_client(os.getenv("SUPABASE_URL"), os.getenv("SUPABASE_SERVICE_KEY"))
try:
    r = SB.table("welfare_services").select("id", count="exact").limit(1).execute()
    print(f"✅ Supabase 연결 OK (현재 welfare_services: {r.count:,}행)")
except Exception as e:
    print(f"❌ Supabase: {e}\n   schema.sql을 먼저 실행하세요.")


✅ 필수 키 OK
✅ Supabase 연결 OK (현재 welfare_services: 15,525행)


## 2. API 클라이언트

In [2]:
BASE = "https://api.odcloud.kr/api/gov24/v3"

def _key():
    raw = os.getenv("PUBLIC_SERVICE_API_KEY", "")
    return unquote(raw) if "%" in raw else raw

def fetch_service_list(per_page=100, max_pages=50, sleep=0.3):
    """공공서비스 목록 - 전체 페이지 순회"""
    all_items = []
    for page in range(1, max_pages + 1):
        params = {"serviceKey": _key(), "page": page, "perPage": per_page, "returnType": "JSON"}
        r = requests.get(f"{BASE}/serviceList", params=params, timeout=30)
        r.raise_for_status()
        body = r.json()
        items = body.get("data", []) or []
        total = body.get("totalCount", 0)
        all_items.extend(items)
        print(f"  page {page:2d}: +{len(items)} (누적 {len(all_items):,}/{total:,})")
        if len(items) < per_page or len(all_items) >= total:
            break
        time.sleep(sleep)
    return all_items

def fetch_support_condition(service_id: str):
    """단일 서비스 지원조건 (JA**** 코드 포함)"""
    params = {"serviceKey": _key(), "page": 1, "perPage": 1, "returnType": "JSON",
              "cond[서비스ID::EQ]": service_id}
    r = requests.get(f"{BASE}/supportConditions", params=params, timeout=20)
    r.raise_for_status()
    data = r.json().get("data", [])
    return data[0] if data else {}

print("API 함수 등록")


API 함수 등록


## 3. 정형화 함수

In [3]:
def _s(v):
    if v is None: return None
    s = str(v).strip()
    return s if s else None

def normalize_service(raw: dict) -> dict:
    """공공서비스 → welfare_services 행"""
    return {
        "source": "행정안전부",
        "service_id":         str(raw.get("서비스ID","")).strip(),
        "service_name":       _s(raw.get("서비스명")),
        "service_summary":    _s(raw.get("서비스목적요약")),
        "agency_name":        _s(raw.get("소관기관명")),
        "agency_type":        _s(raw.get("소관기관유형")),
        "department":         _s(raw.get("부서명")),
        "support_type":       _s(raw.get("지원유형")),
        "user_type":          _s(raw.get("사용자구분")),
        "service_field":      _s(raw.get("서비스분야")),
        "target_description": _s(raw.get("지원대상")),
        "selection_criteria": _s(raw.get("선정기준")),
        "support_content":    _s(raw.get("지원내용")),
        "apply_method":       _s(raw.get("신청방법")),
        "apply_deadline":     _s(raw.get("신청기한")),
        "receiving_agency":   _s(raw.get("접수기관")),
        "contact":            _s(raw.get("전화문의")),
        "detail_url":         _s(raw.get("상세조회URL")),
        "region_sido": None, "region_sigungu": None,
        "life_stages": None, "interest_themes": None,
        "raw_data": raw,
        "fetched_at": datetime.now(timezone.utc).isoformat(),
    }

def _b(v):
    return str(v).strip().lower() in {"y","1","true","t","해당"} if v else False

def _int_or_none(v):
    try: return int(v) if v not in (None,"") else None
    except (TypeError, ValueError): return None

def normalize_conditions(service_id: str, raw: dict) -> dict:
    """supportConditions JA코드 → boolean 컬럼"""
    return {
        "service_id": service_id,
        "age_start": _int_or_none(raw.get("JA0110")),
        "age_end":   _int_or_none(raw.get("JA0111")),
        "male_eligible":   _b(raw.get("JA0101")) if raw.get("JA0101") is not None else True,
        "female_eligible": _b(raw.get("JA0102")) if raw.get("JA0102") is not None else True,
        "income_band_50":      _b(raw.get("JA0201")),
        "income_band_75":      _b(raw.get("JA0202")),
        "income_band_100":     _b(raw.get("JA0203")),
        "income_band_200":     _b(raw.get("JA0204")),
        "income_band_over200": _b(raw.get("JA0205")),
        "multi_cultural":         _b(raw.get("JA0401")),
        "north_korean_defector":  _b(raw.get("JA0402")),
        "single_parent":          _b(raw.get("JA0403")),
        "single_household":       _b(raw.get("JA0404")),
        "multi_child":            _b(raw.get("JA0411")),
        "no_house":               _b(raw.get("JA0412")),
        "expecting_parent": _b(raw.get("JA0301")),
        "pregnant":         _b(raw.get("JA0302")),
        "postpartum":       _b(raw.get("JA0303")),
        "farmer":           _b(raw.get("JA0313")),
        "fisher":           _b(raw.get("JA0314")),
        "livestock":        _b(raw.get("JA0315")),
        "forester":         _b(raw.get("JA0316")),
        "elementary":       _b(raw.get("JA0317")),
        "middle_school":    _b(raw.get("JA0318")),
        "high_school":      _b(raw.get("JA0319")),
        "university":       _b(raw.get("JA0320")),
        "employed":         _b(raw.get("JA0326")),
        "unemployed":       _b(raw.get("JA0327")),
        "disabled":         _b(raw.get("JA0328")),
        "veteran":          _b(raw.get("JA0329")),
        "illness":          _b(raw.get("JA0330")),
        "raw_codes": raw,
        "fetched_at": datetime.now(timezone.utc).isoformat(),
    }

print("정형화 함수 등록")


정형화 함수 등록


## 4. Upsert 헬퍼

In [4]:
def chunked(seq, size):
    for i in range(0, len(seq), size):
        yield seq[i:i+size]

def upsert(table: str, rows: list, chunk=200):
    if not rows: return 0
    total = 0
    for batch in chunked(rows, chunk):
        SB.table(table).upsert(batch, on_conflict="service_id").execute()
        total += len(batch)
        if total % 500 == 0 or total == len(rows):
            print(f"  ↑ {table}: {total:,}/{len(rows):,}")
    return total

def log_run(source: str, fetched: int, upserted: int, errors: int, note: str=""):
    try:
        SB.table("ingestion_runs").insert({
            "source": source, "fetched_count": fetched, "upserted_count": upserted,
            "error_count": errors,
            "finished_at": datetime.now(timezone.utc).isoformat(), "note": note[:500],
        }).execute()
    except Exception as e:
        print(f"   (로그 기록 실패: {e})")

print("Upsert 헬퍼 준비")


Upsert 헬퍼 준비


## 5. 적재 실행 — 서비스 목록

In [5]:
print("="*60); print("[1/2] 공공서비스 목록 fetch"); print("="*60)
raw_services = fetch_service_list(per_page=100, max_pages=1000)
print(f"\n수집 완료: {len(raw_services):,}건\n")

print("정형화 + welfare_services upsert")
norm = [normalize_service(r) for r in raw_services if r.get("서비스ID")]
upserted = upsert("welfare_services", norm)
log_run("행정안전부_서비스", len(raw_services), upserted, len(raw_services)-len(norm))


[1/2] 공공서비스 목록 fetch
  page  1: +100 (누적 100/10,962)
  page  2: +100 (누적 200/10,962)
  page  3: +100 (누적 300/10,962)
  page  4: +100 (누적 400/10,962)
  page  5: +100 (누적 500/10,962)
  page  6: +100 (누적 600/10,962)
  page  7: +100 (누적 700/10,962)
  page  8: +100 (누적 800/10,962)
  page  9: +100 (누적 900/10,962)
  page 10: +100 (누적 1,000/10,962)
  page 11: +100 (누적 1,100/10,962)
  page 12: +100 (누적 1,200/10,962)
  page 13: +100 (누적 1,300/10,962)
  page 14: +100 (누적 1,400/10,962)
  page 15: +100 (누적 1,500/10,962)
  page 16: +100 (누적 1,600/10,962)
  page 17: +100 (누적 1,700/10,962)
  page 18: +100 (누적 1,800/10,962)
  page 19: +100 (누적 1,900/10,962)
  page 20: +100 (누적 2,000/10,962)
  page 21: +100 (누적 2,100/10,962)
  page 22: +100 (누적 2,200/10,962)
  page 23: +100 (누적 2,300/10,962)
  page 24: +100 (누적 2,400/10,962)
  page 25: +100 (누적 2,500/10,962)
  page 26: +100 (누적 2,600/10,962)
  page 27: +100 (누적 2,700/10,962)
  page 28: +100 (누적 2,800/10,962)
  page 29: +100 (누적 2,900/10,962)
  page 30: 

## 6. 적재 실행 — supportConditions

In [6]:
print("="*60); print("[2/2] supportConditions fetch + 정형화 + upsert"); print("="*60)
print(f"대상: {len(norm):,}건 (서비스 1건당 약 0.15초)\n")

cond_rows = []
errors = 0
START = time.time()
for i, svc in enumerate(norm, 1):
    sid = svc["service_id"]
    try:
        raw_cond = fetch_support_condition(sid)
        if raw_cond:
            cond_rows.append(normalize_conditions(sid, raw_cond))
    except Exception as e:
        errors += 1
        if errors <= 5: print(f"   ⚠️ {sid}: {e}")
    if i % 100 == 0 or i == len(norm):
        elapsed = time.time() - START
        rem = elapsed / i * (len(norm) - i)
        print(f"  [{i:>4}/{len(norm):,}] 조건 {len(cond_rows):,} | 에러 {errors} | 경과 {elapsed/60:.1f}m | 남음 {rem/60:.1f}m")
    time.sleep(0.12)

upserted_c = upsert("welfare_support_conditions", cond_rows)
log_run("행정안전부_지원조건", len(cond_rows), upserted_c, errors)


[2/2] supportConditions fetch + 정형화 + upsert
대상: 10,962건 (서비스 1건당 약 0.15초)

  [ 100/10,962] 조건 100 | 에러 0 | 경과 0.9m | 남음 93.0m
  [ 200/10,962] 조건 200 | 에러 0 | 경과 1.7m | 남음 92.8m
  [ 300/10,962] 조건 300 | 에러 0 | 경과 2.6m | 남음 91.0m
  [ 400/10,962] 조건 400 | 에러 0 | 경과 3.4m | 남음 90.3m
  [ 500/10,962] 조건 500 | 에러 0 | 경과 4.2m | 남음 88.5m
  [ 600/10,962] 조건 600 | 에러 0 | 경과 5.0m | 남음 87.0m
  [ 700/10,962] 조건 700 | 에러 0 | 경과 5.9m | 남음 85.8m
  [ 800/10,962] 조건 800 | 에러 0 | 경과 6.7m | 남음 85.0m
  [ 900/10,962] 조건 900 | 에러 0 | 경과 7.5m | 남음 83.9m
  [1000/10,962] 조건 1,000 | 에러 0 | 경과 8.4m | 남음 83.3m
  [1100/10,962] 조건 1,100 | 에러 0 | 경과 9.2m | 남음 82.2m
  [1200/10,962] 조건 1,200 | 에러 0 | 경과 10.0m | 남음 81.4m
  [1300/10,962] 조건 1,300 | 에러 0 | 경과 10.8m | 남음 80.4m
  [1400/10,962] 조건 1,400 | 에러 0 | 경과 11.7m | 남음 80.2m
  [1500/10,962] 조건 1,500 | 에러 0 | 경과 12.5m | 남음 79.1m
  [1600/10,962] 조건 1,600 | 에러 0 | 경과 13.4m | 남음 78.2m
  [1700/10,962] 조건 1,700 | 에러 0 | 경과 14.2m | 남음 77.1m
  [1800/10,962] 조건 1,800 | 에러 0 | 경

## 7. 검증

In [7]:
r1 = SB.table("welfare_services").select("id", count="exact").eq("source","행정안전부").limit(1).execute()
r2 = SB.table("welfare_support_conditions").select("id", count="exact").limit(1).execute()
print(f"welfare_services (행정안전부): {r1.count:,} 행")
print(f"welfare_support_conditions   : {r2.count:,} 행")

# 샘플
sample = (SB.table("welfare_services")
          .select("service_name, agency_name, support_content")
          .eq("source","행정안전부").ilike("service_name","%기초연금%").limit(3).execute())
print("\n샘플 (기초연금 검색):")
for s in sample.data:
    print(f"  - {s['service_name']} ({s['agency_name']})")


welfare_services (행정안전부): 10,967 행
welfare_support_conditions   : 15,525 행

샘플 (기초연금 검색):
  - 기초연금 수급자 확인서 (보건복지부)
  - (경상남도 사천시)100세이상 기초연금 어르신 장제비 지원 (경상남도 사천시)
  - 기초연금 (보건복지부)


## 다음 단계
→ **02_ingest_local.ipynb** 실행 (지자체 데이터 적재)
